# Experimental Gamma Candidate Builder

This notebook is for trying the more Phase-B-like gamma update without touching the current official `gamma_mixed_final.npy`.

It writes everything to `Outputs/meanfield_active/gamma_experiments/...`:

- every saved iteration checkpoint
- a final candidate gamma
- JSON metadata
- a CSV history table

Only promote a candidate into the official path after it has been checked against D2.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from FRASCI.mfa.gamma_experiment import (
    evaluate_gamma_candidate_d2,
    run_gamma_experiment,
)

FCIDUMP = ROOT / "data" / "fcidump_cycle_6"
DETS = ROOT / "data" / "dets.npz"
CURRENT_GAMMA = ROOT / "Outputs" / "meanfield_active" / "outs_extraction_autodets" / "gamma_mixed_final.npy"
EXPERIMENT_DIR = ROOT / "Outputs" / "meanfield_active" / "gamma_experiments" / "damped_mixing_010"

print("FCIDUMP:", FCIDUMP)
print("DETS:", DETS)
print("Current gamma exists:", CURRENT_GAMMA.exists())
print("Experiment dir:", EXPERIMENT_DIR)

## Run The Candidate Gamma Loop

Defaults are intentionally conservative:

- start from the current generated gamma if it exists
- use smaller mixing (`0.10`) than the previous regenerated run
- allow adaptive damping down to `0.02`
- save every iteration so we can test checkpoints later

In [ ]:
TRIMCI_CONFIG = {
    "threshold": 0.06,
    "max_final_dets": "auto",
    "max_rounds": 2,
    "num_runs": 1,
    "pool_build_strategy": "heat_bath",
    "verbose": False,
}

START_GAMMA = str(CURRENT_GAMMA) if CURRENT_GAMMA.exists() else None

metadata = run_gamma_experiment(
    fcidump_path=str(FCIDUMP),
    dets_npz_path=str(DETS),
    output_dir=str(EXPERIMENT_DIR),
    config=TRIMCI_CONFIG,
    max_iter=30,
    tol=1e-4,
    mixing=0.10,
    min_mixing=0.02,
    adaptive=True,
    start_gamma_path=START_GAMMA,
    save_every=1,
)

print(json.dumps({
    "converged": metadata["converged"],
    "iterations_completed": metadata["iterations_completed"],
    "final_gamma_path": metadata["final_gamma_path"],
    "final_delta": metadata["history"][-1]["max_delta_gamma"],
    "final_raw_delta": metadata["history"][-1]["raw_max_delta_gamma"],
    "final_mixing": metadata["final_mixing"],
}, indent=2))

## Inspect The History

In [ ]:
import csv

history_path = EXPERIMENT_DIR / "gamma_experiment_history.csv"
with history_path.open() as fh:
    rows = list(csv.DictReader(fh))

for row in rows[-10:]:
    print(row)

## Optional D2 Check

Leave `RUN_D2_CHECK = False` while experimenting. Turn it on after the gamma loop finishes if you want to test the final candidate against the same D2 h1diag-best settings.

In [ ]:
RUN_D2_CHECK = False

if RUN_D2_CHECK:
    final_gamma = EXPERIMENT_DIR / "gamma_mixed_final_candidate.npy"
    d2_output = ROOT / "Outputs" / "mfa" / "D2_gamma_experiment_h1diag_best"
    d2_config = {
        "threshold": 0.01,
        "max_final_dets": 1000,
        "max_rounds": 2,
        "num_runs": 1,
        "pool_build_strategy": "heat_bath",
        "verbose": False,
    }
    results = evaluate_gamma_candidate_d2(
        fcidump_path=str(FCIDUMP),
        gamma_path=str(final_gamma),
        ref_dets_path=str(DETS),
        output_dir=str(d2_output),
        trimci_config=d2_config,
        partition="h1diag",
    )
    print(json.dumps({
        "E_total": results["E_total"],
        "error_vs_reference": results["error_vs_reference"],
        "fragment_n_dets": results["fragment_n_dets"],
        "total_dets": results["total_dets"],
        "output_dir": str(d2_output),
    }, indent=2))
else:
    print("D2 check skipped. Set RUN_D2_CHECK = True to test the final gamma candidate.")

## Promoting A Candidate Later

Do not copy anything automatically from this notebook. If a checkpoint gives better D2 values, then promote that `.npy` manually after saving the old official file.